# Initial EDA

This notebook explores the raw United Nations General Debate Corpus 1946-2025 (UNGDC) dataset, provided by Harvard Dataverse: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/0TJX8Y.

## Initial set up

In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 150)

DATA_DIR = Path("../data/TXT")

print(f"Data directory exists: {DATA_DIR.exists()}")
print(f"Data directory: {DATA_DIR.resolve()}")

In [ ]:
# Discover files and extract metadata
txt_files = sorted(DATA_DIR.rglob("*.txt"))

print(f"Total text files found: {len(txt_files):,}")
print("\nExample paths:")

for path in txt_files[:10]:
    print(path)

In [ ]:
# Build a metadata table from the file paths
records = []

for path in txt_files:
    filename_match = re.match(
        r"([A-Z]{2,3})_(\d+)_(\d{4})\.txt$",
        path.name
    )

    folder_match = re.search(
        r"Session\s+(\d+)\s*-\s*(\d{4})",
        str(path.parent)
    )

    if filename_match:
        country_code = filename_match.group(1)
        session = int(filename_match.group(2))
        year = int(filename_match.group(3))
    else:
        country_code = None
        session = None
        year = None

    records.append({
        "file_path": str(path),
        "filename": path.name,
        "country_code": country_code,
        "session": session,
        "year": year,
        "folder": path.parent.name,
        "folder_session": int(folder_match.group(1)) if folder_match else None,
        "folder_year": int(folder_match.group(2)) if folder_match else None,
    })

metadata = pd.DataFrame(records)

display(metadata.head())

In [ ]:
print("Files with unparsed country/session/year:")
display(metadata[metadata["country_code"].isna()])

print("Filename vs. folder metadata mismatches:")
display(
    metadata[
        (metadata["year"] != metadata["folder_year"]) |
        (metadata["session"] != metadata["folder_session"])
    ]
)

The output above shows that data/TXT/Session 47 - 1992/.DS_Store-to-UTF-8.txt is the only file that did not match the regex pattern r"([A-Z]{2,3})_(\d+)_(\d{4})\.txt$". This is expected and this file will be excluded from any further analysis.

In [ ]:
# Keep only files with successfully parsed metadata
metadata = metadata.dropna(
    subset=["country_code", "session", "year"]
).copy()

metadata["year"] = metadata["year"].astype(int)
metadata["session"] = metadata["session"].astype(int)

print(f"Valid speeches: {len(metadata):,}")
print(f"Unique countries/entities: {metadata['country_code'].nunique()}")

## Read the text files

In [ ]:
def read_text_file(path):
    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return ""

metadata["text"] = metadata["file_path"].apply(
    lambda x: read_text_file(Path(x))
)

metadata["char_count"] = metadata["text"].str.len()
metadata["word_count"] = metadata["text"].str.split().str.len()
metadata["line_count"] = metadata["text"].str.count(r"\n") + 1

metadata["is_empty"] = metadata["text"].str.strip().eq("")

print(f"Successfully loaded: {(~metadata['is_empty']).sum():,}")
print(f"Empty files: {metadata['is_empty'].sum():,}")

display(
    metadata[
        ["country_code", "year", "word_count", "char_count", "line_count"]
    ].head(10)
)

In [ ]:
sample = metadata.loc[metadata["word_count"].idxmax()]

print(f"Country: {sample['country_code']}")
print(f"Year: {sample['year']}")
print(f"Words: {sample['word_count']:,}")
print("\nFirst 2,000 characters:\n")
print(sample["text"][:2000])

## Overall dataset statistics

In [ ]:
print(f"Total speeches: {len(metadata):,}")
print(f"Unique countries: {metadata['country_code'].nunique():,}")
print(f"Years covered: {metadata['year'].min()}–{metadata['year'].max()}")
print(f"Unique sessions: {metadata['session'].nunique():,}")

display(
    metadata[["word_count", "char_count", "line_count"]].describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    ).round(1)
)

### Duplicate check

In [ ]:
metadata["text_normalized"] = (
    metadata["text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

metadata["text_hash"] = pd.util.hash_pandas_object(
    metadata["text_normalized"],
    index=False
).astype(str)

duplicate_docs = metadata[
    metadata.duplicated("text_hash", keep=False)
].sort_values(["text_hash", "year"])

print(f"Documents involved in exact-text duplicates: {len(duplicate_docs):,}")

display(
    duplicate_docs[
        ["country_code", "year", "filename", "word_count"]
    ].head(30)
)

### Coverage over time and space

In [ ]:
# Coverage over time
speeches_per_year = (
    metadata.groupby("year")
    .size()
    .reset_index(name="speech_count")
    .sort_values("year")
)

plt.figure(figsize=(14, 5))
sns.lineplot(data=speeches_per_year, x="year", y="speech_count", marker="o")
plt.title("Number of UN General Debate Speeches per Year")
plt.xlabel("Year")
plt.ylabel("Number of speeches")
plt.grid(alpha=0.3)
plt.show()

display(speeches_per_year.describe().round(1))

In [ ]:
# Countries per year
countries_per_year = (
    metadata.groupby("year")["country_code"]
    .nunique()
    .reset_index(name="country_count")
)

plt.figure(figsize=(14, 5))
sns.lineplot(data=countries_per_year, x="year", y="country_count", marker="o")
plt.title("Number of Countries Represented per Year")
plt.xlabel("Year")
plt.ylabel("Unique countries")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Check whether each country has exactly one speech per year
country_year_counts = (
    metadata.groupby(["country_code", "year"])
    .size()
    .reset_index(name="speech_count")
)

duplicates = country_year_counts[
    country_year_counts["speech_count"] > 1
].sort_values(["year", "country_code"])

print(f"Total country-year combinations: {len(country_year_counts):,}")
print(f"Country-year combinations with multiple speeches: {len(duplicates):,}")

if duplicates.empty:
    print("Confirmed: Every country has at most one speech per year.")
else:
    print("Multiple speeches found for these country-year combinations:")
    display(duplicates)

### Country coverage

In [ ]:
country_summary = (
    metadata.groupby("country_code")
    .agg(
        speech_count=("filename", "count"),
        first_year=("year", "min"),
        last_year=("year", "max"),
        avg_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        years_present=("year", "nunique")
    )
    .sort_values("speech_count", ascending=False)
)

display(country_summary.head(20).round(1))

### Speech length distribution

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(metadata["word_count"], bins=50, kde=True)
plt.title("Distribution of Speech Lengths")
plt.xlabel("Number of words")
plt.ylabel("Number of speeches")
plt.show()

print("Median speech length:", metadata["word_count"].median())
print("Mean speech length:", round(metadata["word_count"].mean()))
print("Shortest speech:", metadata["word_count"].min())
print("Longest speech:", metadata["word_count"].max())

In [ ]:
print("10 shortest speeches:")
display(
    metadata.nsmallest(10, "word_count")[
        ["country_code", "year", "word_count", "filename"]
    ]
)

print("10 longest speeches:")
display(
    metadata.nlargest(10, "word_count")[
        ["country_code", "year", "word_count", "filename"]
    ]
)

### Initial text-quality checks

In [ ]:
# Non-ASCII characters
metadata["non_ascii_count"] = metadata["text"].apply(
    lambda x: sum(not c.isascii() for c in x)
)

# Replacement characters can indicate encoding problems
metadata["replacement_char_count"] = metadata["text"].str.count("\ufffd")

# Words containing digits
metadata["digit_token_count"] = metadata["text"].apply(
    lambda x: sum(bool(re.search(r"\d", token)) for token in x.split())
)

# Tokens with unusual punctuation/symbols
metadata["unusual_token_count"] = metadata["text"].apply(
    lambda x: sum(
        bool(re.search(r"[^A-Za-z0-9'-]", token))
        for token in x.split()
    )
)

print(
    metadata[
        [
            "country_code",
            "year",
            "word_count",
            "non_ascii_count",
            "replacement_char_count",
            "digit_token_count",
            "unusual_token_count"
        ]
    ].describe().round(2)
)

In [ ]:
print(
    metadata.nlargest(10, "replacement_char_count")[
        ["country_code", "year", "replacement_char_count", "filename"]
    ]
)

In [ ]:
# Inspect the document containing replacement characters
vat_2025 = metadata[
    metadata["filename"] == "VAT_80_2025.txt"
].iloc[0]["text"]

for match in re.finditer("\ufffd", vat_2025):
    start = max(0, match.start() - 100)
    end = min(len(vat_2025), match.start() + 100)

    print(vat_2025[start:end])
    print("-" * 80)

Deeper inspection of the VAT_80_2025.txt file reveals that the unusual tokens were simply meant to be quotation marks and apostrophes. # TODO - address this issue

## Initial NLP EDA

### Initial Vocabulary Exploration

In [ ]:
import nltk
from nltk.corpus import stopwords

try:
    stop_words = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    stop_words = set(stopwords.words("english"))

# Add common words that are frequent in formal UN speeches
custom_stopwords = {
    "united", "nations", "nation", "assembly", "general",
    "international", "world", "countries", "country",
    "states", "state", "government", "governments",
    "people", "peoples", "members", "member",
    "organization", "organization's", "session",
    "president", "delegation", "representative",
    "speaker", "today", "year", "years", "must",
    "shall", "may", "would", "could", "should",
    "one", "two", "also", "us", "we", "our"
}

stop_words = stop_words.union(custom_stopwords)

In [ ]:
# Tokenize speeches and get most frequent items
def tokenize(text):
    tokens = re.findall(r"[a-zA-Z]{3,}", text.lower())
    return [
        token for token in tokens
        if token not in stop_words
    ]

all_tokens = []

for text in metadata["text"].dropna():
    all_tokens.extend(tokenize(text))

word_counts = Counter(all_tokens)

top_words = pd.DataFrame(
    word_counts.most_common(30),
    columns=["word", "frequency"]
)

display(top_words)

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(
    data=top_words.head(25),
    x="frequency",
    y="word"
)
plt.title("Most Frequent Words Across the Corpus")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.show()

In [ ]:
# Compare vocabulary across countries
def get_country_word_counts(country_code):
    texts = metadata.loc[
        metadata["country_code"] == country_code, "text"
    ].dropna()

    tokens = []
    for text in texts:
        tokens.extend(tokenize(text))

    return Counter(tokens)

# Choose the five most represented countries
selected_countries = country_summary.head(5).index.tolist()

country_word_counts = {
    country: get_country_word_counts(country)
    for country in selected_countries
}

for country, counts in country_word_counts.items():
    print(f"\n{country}")
    print(counts.most_common(15))